## Faiyad Mahabub Part A (Q4 and Q5)

## Q4: Sentence Pobabilities using Bigram Language Models - Python Implementation

### Imports

In [18]:
import nltk
from nltk.lm import MLE, Laplace
import numpy as np
from nltk.lm.preprocessing import padded_everygram_pipeline

The required libraries are imported, which include:
- MLE and Laplace: Provide the unsmoothed and smoothed bigram models from nltk package.
- Numpy: It was used for computing the final sentence probability.
- Padded_everygram_pipeline: It handles sentence padding and n-gram generation automatically.



### Loading Data Into Workspace

In [60]:
# Load file
with open("../Data/Data_3.txt", "r", encoding="utf-8") as file:
    text = file.read()
    
print(text)

Training Corpus
~~~~~~~~~~~~~
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

Calculate sentence probability for the following sentence
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
<s> I read a different book by Danielle </s>


*The Data_3.txt is loaded into our workspace. The output confirms that the Data_3.txt has been imported successfully*

### Preparing Training Data

#### Separating Training Corpus And Test Sentence

In [63]:
# Separate corpus and test sentence
lines = text.splitlines()

corpus = [lines[2].strip(), lines[3].strip(), lines[4].strip()]  # Training sentences
test_sentence = lines[8].strip()  # Test sentence

print("Corpus:", corpus)
print("\nTest:", test_sentence)

Corpus: ['<s> He read a book </s>', '<s> I read a different book </s>', '<s> He read a book by Danielle </s>']

Test: <s> I read a different book by Danielle </s>


*The training corpus and the test sentence are extracted from text data using the .strip() method. The method removes any trailing whitespace or newline characters from each line.*

#### Tokenization for both training and test sentences

In [21]:
# Tokenize by splitting on spaces
tokenized_text = [sent.split() for sent in corpus]
test_tokens = test_sentence.split()

print("Tokenized corpus:\n", tokenized_text)
print("\nTokenized test:\n", test_tokens)
print()

Tokenized corpus:
 [['<s>', 'He', 'read', 'a', 'book', '</s>'], ['<s>', 'I', 'read', 'a', 'different', 'book', '</s>'], ['<s>', 'He', 'read', 'a', 'book', 'by', 'Danielle', '</s>']]

Tokenized test:
 ['<s>', 'I', 'read', 'a', 'different', 'book', 'by', 'Danielle', '</s>']



*The training corpus and the test sentence are tokenized by splitting using Python's .split() method. This approach is appropriate here as the corpus is clean and uniformly formatted with no punctuation ambiguity.*

#### Stripping Sentence Markers

In [22]:
# Strip existing markers in training corpus
clean_tokenized = [[w for w in sent if w not in ['<s>', '</s>']] 
                                for sent in tokenized_text]
print("Cleaned tokenized corpus:\n", clean_tokenized)

Cleaned tokenized corpus:
 [['He', 'read', 'a', 'book'], ['I', 'read', 'a', 'different', 'book'], ['He', 'read', 'a', 'book', 'by', 'Danielle']]


*The existing `<s>` and `</s>` boundary markers were removed from the tokenized corpus before being passed to padded_everygram_pipeline. This is necessary because the pipeline automatically adds its own sentence boundary padding, and retaining the original markers would result in duplicate boundaries during training.*

#### All Bigrams In Test Sentence

In [23]:
test_bigrams = list(nltk.bigrams(test_tokens))
print("\nTest bigrams:\n", test_bigrams)


Test bigrams:
 [('<s>', 'I'), ('I', 'read'), ('read', 'a'), ('a', 'different'), ('different', 'book'), ('book', 'by'), ('by', 'Danielle'), ('Danielle', '</s>')]


*The test sentence bigrams were extracted using nltk.bigrams(), where it is required by both the MLE and Laplace probability computation steps that follow.*

## Model Training & Sentence Probability

### Model Training (unsmoothed)

In [24]:
#using padded everygram pipeline to create  unigrams + bigrams and adds markers
train_data, vocab_data = padded_everygram_pipeline(2, clean_tokenized)

model_mle = MLE(2)
model_mle.fit(train_data, vocab_data)

print(f"Vocabulary size: {len(model_mle.vocab)}")
print("MLE Model N-grams:", model_mle.counts)

Vocabulary size: 11
MLE Model N-grams: <NgramCounter with 2 ngram orders and 39 ngrams>


*The cleaned tokenized corpus was passed to padded_everygram_pipeline with order 2, which generates all unigrams and bigrams and adds sentence boundary padding automatically. The MLE model was then fitted on the resulting training data and vocabulary. The vocabulary size of 11 reflects the 10 unique tokens from the corpus plus an additional `<UNK>`token added automatically by NLTK.*


### MLE Probabilities for All Bigrams

In [25]:
# Calculate unsmoothed probabilities
print("Unsmoothed probabilities:")
mle_probs = []

for w1, w2 in test_bigrams:
    prob = model_mle.score(w2, [w1])
    mle_probs.append(prob)
    print(f"  P({w2}|{w1}) = {prob:.4f}")

Unsmoothed probabilities:
  P(I|<s>) = 0.3333
  P(read|I) = 1.0000
  P(a|read) = 1.0000
  P(different|a) = 0.3333
  P(book|different) = 1.0000
  P(by|book) = 0.3333
  P(Danielle|by) = 1.0000
  P(</s>|Danielle) = 1.0000


*Each bigram probability was computed using model_mle.score(w2, [w1]), which applies the MLE formula P(w|w₋₁) = C(w₋₁w) / C(w₋₁) internally.*

### Overall Sentence Probability

In [26]:
sentence_prob_mle = np.prod(mle_probs)

print(f"Sentence probability = {sentence_prob_mle:.6f}")
print(f"                     = {sentence_prob_mle:.2e}")

Sentence probability = 0.037037
                     = 3.70e-02


*The overall sentence probability was obtained by multiplying all individual bigram probabilities using `np.prod().` The result of 0.037037 (3.70%) matches the manual unsmoothed calculation in section 4.2 exactly.*

### Training Laplace Model (Smoothed Bigram Model)

In [27]:
train_data, vocab_data = padded_everygram_pipeline(2, clean_tokenized)
model_laplace = Laplace(2)
model_laplace.fit(train_data, vocab_data)
print(f"Laplace model trained (V = {len(model_laplace.vocab)})")

Laplace model trained (V = 11)


*The MLE training depleted all of the train_data generated by the padded_everygram_pipeline before. So the pipeline is re-run before training the Laplace model to generate a fresh data stream to the model.*

### Laplace Probabilities for All Bigrams

In [28]:
# Calculate smoothed probabilities
print("Smoothed probabilities:")
laplace_probs = []

for w1, w2 in test_bigrams:
    prob = model_laplace.score(w2, [w1])
    laplace_probs.append(prob)
    print(f"  P({w2}|{w1}) = {prob:.4f}")

Smoothed probabilities:
  P(I|<s>) = 0.1429
  P(read|I) = 0.1667
  P(a|read) = 0.2857
  P(different|a) = 0.1429
  P(book|different) = 0.1667
  P(by|book) = 0.1429
  P(Danielle|by) = 0.1667
  P(</s>|Danielle) = 0.1667


*Each bigram probability was computed using model_laplace.score(w2, [w1]), which applies the smoothed formula P(w|w₋₁) = (C(w₋₁w) + 1) / (C(w₋₁) + V) internally.*

### Overall Sentence Probability using Laplace

In [29]:
# Calculate sentence probability
sentence_prob_laplace = np.prod(laplace_probs)

print(f"Sentence probability = {sentence_prob_laplace:.10f}")
print(f"                     = {sentence_prob_laplace:.2e}")

Sentence probability = 0.0000006427
                     = 6.43e-07


## Q5: Alternative Approach Implementation 

### Imports

In [ ]:
import stanza
import string
import time
import pandas as pd
from collections import Counter

### Loading Text Corpus

In [55]:
with open(r"C:\Asia Pacific University\All Module Notes\Semister-5\Text Analysis & Sentimental Analysis\Assignment\CT107-3-3-TXSA - Group Assignment\Data\Data_1.txt") as f:
    text = f.read()

print("Corpus:")
print(text)

Corpus:
Classification is the task of choosing the correct class label for a given input. In basic
classification tasks, each input is considered in isolation from all other inputs, and the set of labels is defined in advance. The basic classification task has a number of interesting variants. For example, in multiclass classification, each instance may be assigned multiple labels; in open-class classification, the set of labels is not defined in advance; and in sequence classification, a list of inputs are jointly classified.


### Tokenization using Stanza

In [56]:
# Tokenize using Stanza
nlp = stanza.Pipeline(lang='en', processors='tokenize', verbose=False)

start = time.time()
doc = nlp(text)
end = time.time()

stanza_tokens = [token.text for sentence in doc.sentences for token in sentence.tokens]
execution_time_ms = (end - start) * 1000

print("Stanza Tokens:\n")
print(stanza_tokens)
print(f"\nExecution time: {execution_time_ms:.2f} ms")

Stanza Tokens:

['Classification', 'is', 'the', 'task', 'of', 'choosing', 'the', 'correct', 'class', 'label', 'for', 'a', 'given', 'input', '.', 'In', 'basic', 'classification', 'tasks', ',', 'each', 'input', 'is', 'considered', 'in', 'isolation', 'from', 'all', 'other', 'inputs', ',', 'and', 'the', 'set', 'of', 'labels', 'is', 'defined', 'in', 'advance', '.', 'The', 'basic', 'classification', 'task', 'has', 'a', 'number', 'of', 'interesting', 'variants', '.', 'For', 'example', ',', 'in', 'multiclass', 'classification', ',', 'each', 'instance', 'may', 'be', 'assigned', 'multiple', 'labels', ';', 'in', 'open', '-', 'class', 'classification', ',', 'the', 'set', 'of', 'labels', 'is', 'not', 'defined', 'in', 'advance', ';', 'and', 'in', 'sequence', 'classification', ',', 'a', 'list', 'of', 'inputs', 'are', 'jointly', 'classified', '.']

Execution time: 319.60 ms


### Defining Metrics For Stanza

In [57]:
# Metric 1: Total Tokens
total_tokens = len(stanza_tokens)

# Metric 2: Tokens with Punctuation Attached
punct_attached = sum(
    1 for t in stanza_tokens
    if any(p in t for p in string.punctuation) and not all(p in string.punctuation for p in t)
)

# ---------- Metric 3: Unique Tokens ----------
# Metric 3a: Unique Tokens (Case-Sensitive) 
unique_case_sensitive = len(set(stanza_tokens))

# Metric 3b: Unique Tokens (Lowercased)
unique_lowercased = len(set(t.lower() for t in stanza_tokens))
vocab_inflation   = unique_case_sensitive - unique_lowercased

# Metric 3c: Case-Sensitive Duplicate Pairs
token_counter = Counter(stanza_tokens)
case_pairs = [
    (t, t.lower()) for t in token_counter
    if t[0].isupper() and t.lower() in token_counter
]

# Metric 4: Hyphenated Words Split Incorrectly
hyphen_words = [word for word in text.split() if '-' in word]
hyphen_split = sum(
    1 for hw in hyphen_words
    if hw not in stanza_tokens
)


# Metric 6: Punctuation Type Breakdown
punct_types = {
    'Periods (.)':    [t for t in stanza_tokens if t == '.'],
    'Commas (,)':     [t for t in stanza_tokens if t == ','],
    'Semicolons (;)': [t for t in stanza_tokens if t == ';'],
    'Hyphens (-)':    [t for t in stanza_tokens if t == '-'],
}



### Full Summary Output of the Metrics

In [58]:
def punct_types_summary(punct_types):
    return ", ".join(f"{ptype}: {len(tokens)}" for ptype, tokens in punct_types.items())

def print_token_metrics(method_name, total, punct_attached, hyphen_split, unique, unique_lowercased, vocab_inflation, case_pairs, punct_types, execution_time_ms):
    print(f"--- {method_name} ---")
    print(f"Total Tokens:                       {total}")
    print(f"Tokens with Punctuation Attached:   {punct_attached}")
    print(f"Unique Tokens (Case-Sensitive):     {unique}")
    print(f"Unique Tokens (Lowercased):         {unique_lowercased}")
    print(f"Vocabulary Inflation Due to Case:   {vocab_inflation}")
    print(f"Case-Sensitive Duplicate Pairs:     {len(case_pairs)}")
    print(f"Hyphenated Words Split Incorrectly: {hyphen_split}")
    print(f"Punctuation Type Breakdown:         {punct_types_summary(punct_types)}")
    print(f"Execution Time (ms):                {execution_time_ms:.2f} ms\n")

print_token_metrics(
    "Stanza",
    total_tokens,
    punct_attached,
    hyphen_split,
    unique_case_sensitive,
    unique_lowercased,
    vocab_inflation,
    case_pairs,
    punct_types,
    execution_time_ms
)

--- Stanza ---
Total Tokens:                       96
Tokens with Punctuation Attached:   0
Unique Tokens (Case-Sensitive):     54
Unique Tokens (Lowercased):         50
Vocabulary Inflation Due to Case:   4
Case-Sensitive Duplicate Pairs:     4
Hyphenated Words Split Incorrectly: 1
Punctuation Type Breakdown:         Periods (.): 4, Commas (,): 6, Semicolons (;): 2, Hyphens (-): 1
Execution Time (ms):                319.60 ms

